# Patient tracks

Longitudinal clinical measures, medications, and 1-, 3-, and 5-year predicted risk for one example patient in each of T2D, CKD, and MASLD.

Clinical inputs come from `data/`, including the full A1c, eGFR, and FIB-4 histories. Absolute risks are `pred_risk_1yr`, `pred_risk_3yr`, and `pred_risk_5yr` from `outputs/main/absolute_risks/`.


In [ ]:
library(tidyverse)
library(here)

source(here("helpers/plot_patient_tracks.r"))

fig_dir <- here("outputs/figures")
dir.create(fig_dir, showWarnings = FALSE, recursive = TRUE)

# plot_patient_tracks() expects risk_1y / risk_3y / risk_5y.
risk_df_from_fold_baseline <- function(res) {
  oof <- res$oof_predictions
  data.frame(
    StudyID = as.character(oof$StudyID),
    risk_1y = as.numeric(oof$pred_risk_1yr),
    risk_3y = as.numeric(oof$pred_risk_3yr),
    risk_5y = as.numeric(oof$pred_risk_5yr),
    stringsAsFactors = FALSE
  )
}

as_track <- function(df, date_col, value_col) {
  df %>%
    transmute(
      StudyID = as.character(StudyID),
      CLIN_DATE = as.Date(.data[[date_col]]),
      CLIN_VAL = readr::parse_number(as.character(.data[[value_col]]))
    ) %>%
    filter(!is.na(CLIN_DATE), !is.na(CLIN_VAL)) %>%
    group_by(StudyID, CLIN_DATE) %>%
    summarize(CLIN_VAL = median(CLIN_VAL), .groups = "drop")
}


In [ ]:
in_dir <- here("data")

meta <- read.csv(file.path(in_dir, "meta.csv"), row.names = 1) %>%
  mutate(StudyID = as.character(StudyID))

bmi <- read.csv(file.path(in_dir, "bmi.csv")) %>%
  as_track("MEASUREMENT_DATE", "BMI")

bp <- read.csv(file.path(in_dir, "sbp.csv")) %>%
  as_track("MEASUREMENT_DATE", "Systolic")

a1c <- read.csv(file.path(in_dir, "a1c_history.csv")) %>%
  as_track("RESULT_DATE", "Result_C")

rx <- read.csv(file.path(in_dir, "rx.csv")) %>%
  transmute(
    StudyID = as.character(StudyID),
    CLIN_DATE = as.Date(RX_DATE),
    GENERIC_NM = stringr::str_trim(GENERIC_NM)
  ) %>%
  filter(!is.na(CLIN_DATE), !is.na(GENERIC_NM), GENERIC_NM != "")

egfr <- read.csv(file.path(in_dir, "egfr_history.csv")) %>%
  as_track("RESULT_DATE", "egfr")

fib4 <- read.csv(file.path(in_dir, "fib4_history.csv")) %>%
  as_track("fib4_date", "fib4")

abs_risk_dir <- here("outputs/main/absolute_risks")

load_risk_pair <- function(prefix, label) {
  list(
    label = label,
    soc = risk_df_from_fold_baseline(readRDS(file.path(
      abs_risk_dir, paste0(prefix, "marker_fold_baseline_risk.rds")
    ))),
    nmr = risk_df_from_fold_baseline(readRDS(file.path(
      abs_risk_dir, paste0(prefix, "marker_nmr_fold_baseline_risk.rds")
    )))
  )
}

risks <- list(
  t2d   = load_risk_pair("t2d_a1c_", "T2D"),
  ckd   = load_risk_pair("ckd_egfr_", "CKD"),
  masld = load_risk_pair("nafld_fib4_", "MASLD")
)


In [ ]:
clinical_specs <- list(
  "BMI" = list(
    y_limits = c(20, 50),
    bands = data.frame(
      ymin  = c(-Inf, 18.5, 25, 30, 35, 40),
      ymax  = c(18.5, 25, 30, 35, 40, Inf),
      label = c("Underweight", "Normal", "Overweight", "Obese I", "Obese II", "Obese III"),
      fill  = c("#e3f2fd", "#e8f5e9", "#fffde7", "#ffe0b2", "#ffccbc", "#ef9a9a")
    ),
    lines = c(18.5, 25, 30, 35, 40),
    label_bands = TRUE,
    ylabel_pos = c(16.5, 21.5, 27.5, 32.5, 37.5, 45)
  ),
  "SBP" = list(
    y_limits = c(80, 170),
    bands = data.frame(
      ymin  = c(-Inf, 120, 130, 140),
      ymax  = c(120, 130, 140, Inf),
      label = c("Normal", "Elevated", "Stage 1", "Stage 2"),
      fill  = c("#e8f5e9", "#fffde7", "#ffe0b2", "#ef9a9a")
    ),
    lines = c(120, 130, 140),
    label_bands = TRUE,
    ylabel_pos = c(100, 125, 135, 170)
  ),
  "HbA1c (%)" = list(
    y_limits = c(5.0, 8),
    bands = data.frame(
      ymin  = c(-Inf, 5.7, 6.5),
      ymax  = c(5.7, 6.5, Inf),
      label = c("Normal", "Prediabetes", "Diabetes"),
      fill  = c("#e8f5e9", "#ffe0b2", "#ef9a9a")
    ),
    lines = c(5.7, 6.5),
    label_bands = TRUE,
    ylabel_pos = c(5.2, 6.1, 6.8)
  ),
  "eGFR (mL/min/1.73m2)" = list(
    y_limits = c(0, 120),
    bands = data.frame(
      ymin  = c(-Inf, 15, 30, 45, 60, 90),
      ymax  = c(15, 30, 45, 60, 90, Inf),
      label = c("G5 (failure)", "G4 (severe)", "G3b", "G3a", "G2 (mild)", "G1 (normal)"),
      fill  = c("#b71c1c", "#ef9a9a", "#ffe0b2", "#fffde7", "#e8f5e9", "#e3f2fd"),
      stringsAsFactors = FALSE
    ),
    lines = c(15, 30, 45, 60, 90),
    label_bands = TRUE,
    ylabel_pos = c(7, 22, 37, 52, 75, 105)
  ),
  "FIB-4" = list(
    y_limits = c(0, 5),
    bands = data.frame(
      ymin  = c(-Inf, 1.3, 2.67),
      ymax  = c(1.3, 2.67, Inf),
      label = c("Low risk", "Indeterminate", "High risk"),
      fill  = c("#e8f5e9", "#fffde7", "#ef9a9a"),
      stringsAsFactors = FALSE
    ),
    lines = c(1.3, 2.67),
    label_bands = TRUE,
    ylabel_pos = c(0.65, 2.0, 3.8)
  )
)

plot_tracks <- function(patient_id, risk_pair, date_start, date_end, y_limits = list(), ...) {
  specs <- clinical_specs
  for (nm in names(y_limits)) specs[[nm]]$y_limits <- y_limits[[nm]]
  plot_patient_tracks(
    risk_name      = risk_pair$label,
    patient_id     = patient_id,
    meta           = meta,
    rx             = rx,
    df_soc         = risk_pair$soc,
    df_nmr         = risk_pair$nmr,
    risk_horizons  = c(1, 3, 5),
    date_start     = date_start,
    date_end       = date_end,
    use_thresholds = TRUE,
    clinical_specs = specs,
    show_placeholder_risk_panels = TRUE,
    ...
  ) &
    theme(axis.title.y = element_text(margin = margin(r = 12)))
}


## Type 2 diabetes


In [ ]:
options(repr.plot.width = 5.5, repr.plot.height = 7)

p_t2d <- plot_tracks(
  "ID1025889", risks$t2d,
  date_start = "2018-01-01",
  date_end   = "2024-12-31",
  y_limits   = list(BMI = c(20, 50), `HbA1c (%)` = c(5, 10)),
  bmi = bmi, bp = bp, a1c = a1c
)

p_t2d
ggsave(
  file.path(fig_dir, "patient_tracks_t2d.png"),
  p_t2d, width = 5.5, height = 6.5
)

## CKD


In [ ]:
options(repr.plot.width = 5.5, repr.plot.height = 6.5)

p_ckd <- plot_tracks(
  "ID1127460", risks$ckd,
  date_start = "2017-01-01",
  date_end   = "2024-12-31",
  y_limits   = list(BMI = c(20, 35), `eGFR (mL/min/1.73m2)` = c(0, 120)),
  bmi = bmi, bp = bp, egfr = egfr
)

p_ckd
ggsave(
  file.path(fig_dir, "patient_tracks_ckd.png"),
  p_ckd, width = 5.5, height = 6.5
)

## MASLD


In [ ]:
options(repr.plot.width = 5.5, repr.plot.height = 6.5)

p_masld <- plot_tracks(
  "ID1076631", risks$masld,
  date_start = "2018-01-01",
  date_end   = "2024-12-31",
  y_limits   = list(BMI = c(20, 50), `FIB-4` = c(0, 20)),
  bmi = bmi, bp = bp, fib4 = fib4
)

p_masld
ggsave(
  file.path(fig_dir, "patient_tracks_masld.png"),
  p_masld, width = 5.5, height = 6.5
)